# HybridGraphFNet — NeuroGraph HCP-Task Benchmark

**Task:** Multi-class Graph-Level Classification (7 Task fMRI States) on brain functional connectomes ($N=1,000$ ROIs).

**Architecture:** HybridGraphFNet (DenseGCN + SpectralMixMH + Learned Per-Node Gating + GatedPooling readout).

**Baselines Reference (from Wang et al., 2025 BrainMAP, arXiv:2412.17404):**
- GCN: 86.29 ± 0.98%
- GAT: 85.60 ± 1.26%
- GraphSAGE: 84.49 ± 0.57%
- ResGCN: 93.75 ± 0.35%
- GraphGPS: 92.13 ± 2.00%
- Graph-Mamba: 94.17 ± 0.86%
- **BrainMAP (SOTA)**: 94.74 ± 0.07%


In [1]:
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 946.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time, os, gc, csv
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import NeuroGraphDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def set_seed(s):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    np.random.seed(s); torch.backends.cudnn.deterministic = True

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Load NeuroGraph HCP-Task & Inspect

In [3]:
DATA_ROOT = './data'
CACHE_DIR = './eigenbasis_cache_hcp_task'
TRUNC_K   = 64

# Load NeuroGraph HCPTask dataset
full_dataset = NeuroGraphDataset(root=DATA_ROOT, name='HCPTask')

NUM_CLASSES = full_dataset.num_classes if hasattr(full_dataset, 'num_classes') else len(set(d.y.item() if d.y.dim()==0 else d.y[0].item() for d in full_dataset))

sample = full_dataset[0]
HAS_FEATURES = sample.x is not None
NODE_FEAT_DIM = sample.x.shape[-1] if HAS_FEATURES else 1
print(f'Node features: {"dim=" + str(NODE_FEAT_DIM) if HAS_FEATURES else "ABSENT"}')

HAS_EDGE_ATTR = sample.edge_attr is not None
EDGE_DIM = sample.edge_attr.shape[-1] if (HAS_EDGE_ATTR and sample.edge_attr.dim() > 1) else (1 if HAS_EDGE_ATTR else 0)
print(f'Edge attr: {"dim=" + str(EDGE_DIM) if HAS_EDGE_ATTR else "absent"}')

all_n = [d.num_nodes for d in full_dataset]
all_e = [d.num_edges for d in full_dataset]
print(f'\nDataset: {len(full_dataset)} graphs, {NUM_CLASSES} classes')
print(f'Nodes: min={min(all_n)}, max={max(all_n)}, mean={np.mean(all_n):.1f}')
print(f'Edges: min={min(all_e)}, max={max(all_e)}, mean={np.mean(all_e):.0f}')

all_labels = []
for d in full_dataset:
    y = d.y.item() if d.y.dim() == 0 else d.y[0].item()
    all_labels.append(int(y))
all_labels = np.array(all_labels)

print(f'\nClass distribution:')
for c in range(NUM_CLASSES):
    n = (all_labels == c).sum()
    print(f'  Class {c}: {n:>5d} ({100*n/len(all_labels):.1f}%)')

Extracting data/HCPTask/raw/8wzz4y17wpxg2stip7iybtmymnybwvma.zip
Processing...
Done!


Node features: dim=400
Edge attr: absent

Dataset: 7443 graphs, 7 classes
Nodes: min=400, max=400, mean=400.0
Edges: min=4764, max=7980, mean=7029

Class distribution:
  Class 0:  1045 (14.0%)
  Class 1:  1086 (14.6%)
  Class 2:  1051 (14.1%)
  Class 3:  1081 (14.5%)
  Class 4:  1043 (14.0%)
  Class 5:  1051 (14.1%)
  Class 6:  1086 (14.6%)


## 2. Train/Val/Test Split (Stratified 80/10/10)

In [4]:
SPLIT_SEED = 42  # Fixed for reproducibility across runs

indices = np.arange(len(full_dataset))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=SPLIT_SEED, stratify=all_labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SPLIT_SEED, stratify=all_labels[temp_idx])

train_ds = full_dataset[train_idx.tolist()]
val_ds   = full_dataset[val_idx.tolist()]
test_ds  = full_dataset[test_idx.tolist()]

print(f'Split sizes: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')

for name, ds in [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]:
    ys = [d.y.item() if d.y.dim()==0 else d.y[0].item() for d in ds]
    counts = np.bincount(ys, minlength=NUM_CLASSES)
    pcts = 100 * counts / len(ys)
    dist = ', '.join(f'C{i}={counts[i]}({pcts[i]:.0f}%)' for i in range(NUM_CLASSES))
    print(f'  {name}: {dist}')

Split sizes: Train=5954, Val=744, Test=745
  Train: C0=836(14%), C1=869(15%), C2=841(14%), C3=865(15%), C4=834(14%), C5=841(14%), C6=868(15%)
  Val: C0=105(14%), C1=108(15%), C2=105(14%), C3=108(15%), C4=104(14%), C5=105(14%), C6=109(15%)
  Test: C0=104(14%), C1=109(15%), C2=105(14%), C3=108(14%), C4=105(14%), C5=105(14%), C6=109(15%)


## 3. Precompute Truncated Laplacian Eigenbasis ($k=64$)

In [5]:
def precompute_eigenbasis_for_split(dataset, split_name, cache_dir, k=64):
    split_dir = os.path.join(cache_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)
    eigh_failures = 0
    total_bytes = 0
    t_start = time.perf_counter()

    for idx in tqdm(range(len(dataset)), desc=f'Precompute {split_name}'):
        data = dataset[idx]
        n = data.num_nodes
        edge_index = data.edge_index
        adj = torch.zeros(n, n)
        adj[edge_index[0], edge_index[1]] = 1.0
        adj = adj + torch.eye(n)

        deg = adj.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        A_norm = D_inv_sqrt @ adj @ D_inv_sqrt
        L = torch.eye(n) - A_norm

        try:
            _, U = torch.linalg.eigh(L)
            max_abs_idx = torch.abs(U).argmax(dim=0)
            signs = torch.sign(U[max_abs_idx, torch.arange(U.size(1))])
            signs[signs == 0] = 1.0
            U = U * signs.unsqueeze(0)
        except Exception as e:
            eigh_failures += 1
            U = torch.eye(n)
            if eigh_failures <= 5:
                print(f'  WARNING: eigh failed graph {idx} (n={n}): {e}')

        k_actual = min(k, n)
        U_trunc = U[:, :k_actual]
        save_path = os.path.join(split_dir, f'{idx}.pt')
        torch.save({'U': U_trunc.clone(), 'n': n, 'k': k_actual}, save_path)
        total_bytes += os.path.getsize(save_path)

    elapsed = time.perf_counter() - t_start
    print(f'  {split_name}: {len(dataset)} graphs, {eigh_failures} failures, '
          f'{elapsed:.1f}s, {total_bytes/1e6:.1f} MB')
    return {'split': split_name, 'eigh_failures': eigh_failures, 'time_s': elapsed, 'disk_mb': total_bytes/1e6}

for split_name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    split_dir = os.path.join(CACHE_DIR, split_name)
    expected = len(ds)
    existing = len([f for f in os.listdir(split_dir) if f.endswith('.pt')]) if os.path.isdir(split_dir) else 0
    if existing >= expected:
        print(f'{split_name}: cache exists ({existing} files), skipping')
    else:
        precompute_eigenbasis_for_split(ds, split_name, CACHE_DIR, k=TRUNC_K)

print('Eigenbasis cache ready.')

Precompute train:   1%|          | 45/5954 [00:00<01:21, 72.52it/s]

Precompute train:   5%|▌         | 312/5954 [00:03<01:04, 86.88it/s]

Precompute train:   7%|▋         | 444/5954 [00:05<00:57, 95.66it/s]

Precompute train:  10%|█         | 625/5954 [00:07<00:56, 93.69it/s]

Precompute train:  12%|█▏        | 705/5954 [00:08<00:57, 90.90it/s]

...

Precompute test:  23%|██▎       | 168/745 [00:01<00:06, 96.03it/s]

Precompute test:  55%|█████▍    | 408/745 [00:04<00:03, 90.81it/s]

Precompute test:  67%|██████▋   | 498/745 [00:05<00:02, 94.59it/s]

Precompute test: 100%|██████████| 745/745 [00:07<00:00, 93.44it/s]

  test: 745 graphs, 6 failures, 8.0s, 77.4 MB
Eigenbasis cache ready.


## 4. Cached Dataset Wrapper & DataLoaders

In [6]:
class CachedEigenbasisDataset:
    def __init__(self, base_dataset, cache_dir, split_name, k_trunc=64):
        self.base = base_dataset
        self.split_dir = os.path.join(cache_dir, split_name)
        self.k_trunc = k_trunc

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        data = self.base[idx].clone()
        cache = torch.load(os.path.join(self.split_dir, f'{idx}.pt'), weights_only=True)
        U = cache['U']
        if U.size(1) < self.k_trunc:
            U = F.pad(U, (0, self.k_trunc - U.size(1)))
        data.cached_U = U

        if data.x is None:
            from torch_geometric.utils import degree
            row = data.edge_index[0]
            data.x = degree(row, num_nodes=data.num_nodes).float().unsqueeze(-1)
        return data

def make_loaders(batch_size=32):
    tl = DataLoader(CachedEigenbasisDataset(train_ds, CACHE_DIR, 'train', TRUNC_K), batch_size=batch_size, shuffle=True)
    vl = DataLoader(CachedEigenbasisDataset(val_ds, CACHE_DIR, 'val', TRUNC_K), batch_size=batch_size, shuffle=False)
    tel = DataLoader(CachedEigenbasisDataset(test_ds, CACHE_DIR, 'test', TRUNC_K), batch_size=batch_size, shuffle=False)
    return tl, vl, tel

_tl, _, _ = make_loaders(4)
_b = next(iter(_tl))
print(f'Batch: x={_b.x.shape}, cached_U={_b.cached_U.shape}, y={_b.y.shape}')
del _tl, _b

Batch: x=torch.Size([1600, 400]), cached_U=torch.Size([1600, 64]), y=torch.Size([4])


## 5. Model Components

- `DenseGCNLayer`: Local spatial message-passing with scalar edge gating support.
- `SpectralMixMH`: Multi-head learned spectral mixing over Laplacian basis $U$.
- `GatedPooling`: Graph-level readout via learned gating.

In [7]:
class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=0):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1) if edge_dim > 0 else None
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None and self.edge_lin is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))


class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))


class GatedPooling(nn.Module):
    """Graph-level pooling via learned gating (preserves project rule)."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, mask):
        scores  = self.gate(x).squeeze(-1)
        scores  = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)

print('Components defined: DenseGCNLayer, SpectralMixMH, GatedPooling')

Components defined: DenseGCNLayer, SpectralMixMH, GatedPooling


## 6. HybridGraphFNet — Graph-Level Model

In [8]:
class HybridGraphFNet_GraphLevel(nn.Module):
    def __init__(self, in_dim=200, hidden_dim=128, num_layers=4, num_classes=7,
                 num_heads=4, lap_k=8, dropout=0.1, edge_dim=0):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
        )
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'local':  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                'global': SpectralMixMH(hidden_dim, num_heads=num_heads),
                'gate':   nn.Linear(hidden_dim, hidden_dim),
                'norm':   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])

        self.pool = GatedPooling(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def compute_A_norm(self, adj, mask):
        B, N, _ = adj.shape
        A_list = []
        for b in range(B):
            n = int(mask[b].sum().item())
            adj_b = adj[b, :n, :n]
            deg = adj_b.sum(dim=1)
            deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
            D_inv_sqrt = torch.diag(deg_inv_sqrt)
            A_norm_b = D_inv_sqrt @ adj_b @ D_inv_sqrt
            A_list.append(F.pad(A_norm_b, (0, N-n, 0, N-n)))
        return torch.stack(A_list)

    def forward(self, data):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        if hasattr(data, 'edge_attr') and data.edge_attr is not None:
            ea = data.edge_attr.float()
            if ea.dim() == 1: ea = ea.unsqueeze(-1)
            edge_attr_dense = to_dense_adj(data.edge_index, data.batch, edge_attr=ea, max_num_nodes=x.size(1))
        else:
            edge_attr_dense = None

        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        A_norm = self.compute_A_norm(adj, mask)

        U, _ = to_dense_batch(data.cached_U.float(), data.batch)
        U = U * mask.unsqueeze(-1)

        x = self.input_proj(x)
        k = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            x_res    = x
            x_local  = layer['local'](x, A_norm, edge_attr_dense)
            x_global = layer['global'](x, U, mask)
            gate     = torch.sigmoid(layer['gate'](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer['norm'](x_res + self.dropout(x_mix))

        x = x * mask.unsqueeze(-1)
        graph_emb = self.pool(x, mask)
        return self.classifier(graph_emb)

set_seed(0)
model = HybridGraphFNet_GraphLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, edge_dim=EDGE_DIM, dropout=0.1,
).to(device)

print(f'Parameters: {count_params(model):,}')
_tl, _, _ = make_loaders(4)
_b = next(iter(_tl)).to(device)
with torch.no_grad():
    logits = model(_b)
print(f'Forward pass OK: logits={logits.shape}')
del _tl, _b

Parameters: 361,992
Forward pass OK: logits=torch.Size([4, 7])


## 7. Evaluation & Gate Health Check

In [9]:
def evaluate_graph_level(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            preds = logits.argmax(dim=-1).cpu()
            labels = batch.y.squeeze(-1) if batch.y.dim() > 1 else batch.y
            all_preds.append(preds)
            all_labels.append(labels.cpu())
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, f1


def check_gate_health(model, val_loader, device):
    model.eval()
    batch = next(iter(val_loader)).to(device)
    with torch.no_grad():
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        U, _ = to_dense_batch(batch.cached_U.float(), batch.batch)
        U = U * mask.unsqueeze(-1)
        x_enc = model.input_proj(x)
        k = min(model.lap_k, U.size(-1))
        x_enc = x_enc + model.pe_encoder(U[:, :, :k] * mask.unsqueeze(-1))

        print('\n--- Gate Health Check ---')
        x_curr = x_enc
        adj = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=x.size(1))
        adj = adj + torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        A_norm = model.compute_A_norm(adj, mask)

        for idx, layer in enumerate(model.layers):
            gate_vals = torch.sigmoid(layer['gate'](x_curr))
            mean_g = gate_vals[mask].mean().item()
            std_g  = gate_vals[mask].std().item()
            print(f'  Layer {idx}: gate mean={mean_g:.4f}, std={std_g:.4f}')
            x_local = layer['local'](x_curr, A_norm)
            x_global = layer['global'](x_curr, U, mask)
            x_mix = gate_vals * x_local + (1 - gate_vals) * x_global
            x_curr = layer['norm'](x_curr + model.dropout(x_mix))
        print('-------------------------\n')
    return False

## 8. Memory & Speed Smoke Test

In [10]:
BATCH_SIZE = 32

_tl, _, _ = make_loaders(BATCH_SIZE)
_model = HybridGraphFNet_GraphLevel(
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4, num_classes=NUM_CLASSES, edge_dim=EDGE_DIM, dropout=0.1,
).to(device)
_model.train()
_opt = optim.AdamW(_model.parameters(), lr=1e-3, weight_decay=1e-4)
_crit = nn.CrossEntropyLoss()

if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()

_b = next(iter(_tl)).to(device)
t0 = time.perf_counter()
logits = _model(_b)
y = _b.y; y = y.squeeze(-1) if y.dim() > 1 else y
loss = _crit(logits, y.long())
loss.backward()
_opt.step()
if device.type == 'cuda': torch.cuda.synchronize()
t_batch = time.perf_counter() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == 'cuda' else 0.0
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if device.type == 'cuda' else 0.0
batches_ep = len(_tl)

print(f'=== SMOKE TEST ===')
print(f'1 batch ({BATCH_SIZE} graphs, N=1,000): {t_batch*1000:.1f} ms')
print(f'Peak VRAM: {peak_gb:.2f} GB / {total_gb:.1f} GB ({100*peak_gb/total_gb:.0f}%)')
print(f'Est. time/epoch: {t_batch*batches_ep:.1f}s, 200 epochs: {t_batch*batches_ep*200/3600:.1f}h')
del _model, _opt, _b, _tl

=== SMOKE TEST ===
1 batch (32 graphs, N=1,000): 228.2 ms
Peak VRAM: 0.47 GB / 15.6 GB (3%)
Est. time/epoch: 42.7s, 200 epochs: 2.4h


## 9. Baseline Comparison Reference

In [11]:
train_y = np.array([d.y.item() if d.y.dim()==0 else d.y[0].item() for d in train_ds])
majority_class = int(np.bincount(train_y).argmax())
majority_pct = 100 * (train_y == majority_class).sum() / len(train_y)

test_y = np.array([d.y.item() if d.y.dim()==0 else d.y[0].item() for d in test_ds])
maj_preds = np.full_like(test_y, majority_class)
maj_acc = accuracy_score(test_y, maj_preds)
maj_f1  = f1_score(test_y, maj_preds, average='macro', zero_division=0)

print(f'=== BASELINE REFERENCE ===')
print(f'Trivial Majority Class: {majority_class} ({majority_pct:.1f}% of train) -> Test Acc: {maj_acc*100:.2f}%')
print(f'\nLiterature Baselines (from Wang et al., 2025 BrainMAP):')
print(f'  GCN:         86.29% ± 0.98%')
print(f'  GAT:         85.60% ± 1.26%')
print(f'  GraphSAGE:   84.49% ± 0.57%')
print(f'  ResGCN:      93.75% ± 0.35%')
print(f'  GraphGPS:    92.13% ± 2.00%')
print(f'  Graph-Mamba: 94.17% ± 0.86%')
print(f'  BrainMAP:    94.74% ± 0.07%')
print(f'Target: Outperform 94.74%')

=== BASELINE REFERENCE ===
Trivial Majority Class: 1 (14.6% of train) -> Test Acc: 14.63%

Literature Baselines (from Wang et al., 2025 BrainMAP):
  GCN:         86.29% ± 0.98%
  GAT:         85.60% ± 1.26%
  GraphSAGE:   84.49% ± 0.57%
  ResGCN:      93.75% ± 0.35%
  GraphGPS:    92.13% ± 2.00%
  Graph-Mamba: 94.17% ± 0.86%
  BrainMAP:    94.74% ± 0.07%
Target: Outperform 94.74%


## 10. Training Loop (Kaggle / Background Ready)

In [12]:
CKPT_EVERY = 5

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_acc, best_epoch, epochs_no_improve, gate_checked):
    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(), 'epoch': epoch, 'best_val_acc': best_val_acc,
                'best_epoch': best_epoch, 'epochs_no_improve': epochs_no_improve, 'gate_checked': gate_checked}, path)

def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, weights_only=False)
    model.load_state_dict(ckpt['model_state']); optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    return ckpt['epoch'], ckpt['best_val_acc'], ckpt['best_epoch'], ckpt['epochs_no_improve'], ckpt['gate_checked']


def train_single_seed(
    seed, in_dim=200, hidden_dim=128, num_layers=4, num_classes=7,
    num_heads=4, lap_k=8, edge_dim=0, dropout=0.1,
    max_epochs=200, patience=30, batch_size=32, lr=1e-3, weight_decay=1e-4,
):
    train_loader, val_loader, test_loader = make_loaders(batch_size)

    train_y = torch.tensor([d.y.item() if d.y.dim()==0 else d.y[0].item() for d in train_ds])
    cc = torch.bincount(train_y, minlength=num_classes).float().clamp(min=1)
    cw = (1.0 / cc); cw = (cw / cw.sum() * num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)

    best_ckpt   = f'best_model_hcp_task_seed{seed}.pt'
    resume_ckpt = f'resume_hcp_task_seed{seed}.pt'

    print('=' * 65)
    print(f'HCP-Task | Seed {seed} | Hidden={hidden_dim} | BS={batch_size} | LR={lr}')
    print('=' * 65)

    set_seed(seed)
    model = HybridGraphFNet_GraphLevel(
        in_dim=in_dim, hidden_dim=hidden_dim, num_layers=num_layers,
        num_classes=num_classes, num_heads=num_heads, lap_k=lap_k, edge_dim=edge_dim,
        dropout=dropout,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-5)
    print(f'Parameters: {count_params(model):,}')

    start_epoch = 1; best_val_acc = 0.0; best_epoch = 0; epochs_no_improve = 0; gate_checked = False

    if os.path.exists(resume_ckpt):
        print(f'\n>>> RESUMING from {resume_ckpt}')
        prev_ep, best_val_acc, best_epoch, epochs_no_improve, gate_checked = \
            load_checkpoint(resume_ckpt, model, optimizer, scheduler)
        start_epoch = prev_ep + 1
        print(f'    Resuming at epoch {start_epoch}')
    else:
        print('No checkpoint — starting fresh.')

    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for epoch in range(start_epoch, max_epochs + 1):
        if epoch == 6 and not gate_checked:
            gc_result = check_gate_health(model, val_loader, device)
            gate_checked = True
            if gc_result:
                optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=weight_decay)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs-epoch, eta_min=1e-5)

        model.train()
        total_loss = 0; optimizer.zero_grad()
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f'Seed {seed} | Ep {epoch}/{max_epochs}', leave=False)

        for step, batch in pbar:
            batch = batch.to(device)
            logits = model(batch)
            y = batch.y; y = y.squeeze(-1) if y.dim() > 1 else y
            loss = criterion(logits, y.long())
            loss.backward()
            total_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad()
            pbar.set_postfix({'loss': f'{total_loss/(step+1):.4f}'})

        scheduler.step()

        val_acc, val_f1 = evaluate_graph_level(model, val_loader, device)
        improved = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc; best_epoch = epoch; epochs_no_improve = 0
            torch.save(model.state_dict(), best_ckpt); improved = ' *BEST*'
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or improved:
            print(f'  Ep {epoch:>3d} | loss={total_loss/len(train_loader):.4f} | '
                  f'val_acc={val_acc:.4f} val_F1={val_f1:.4f} | best={best_val_acc:.4f}@ep{best_epoch}{improved}')

        if epoch % CKPT_EVERY == 0:
            save_checkpoint(resume_ckpt, model, optimizer, scheduler,
                           epoch, best_val_acc, best_epoch, epochs_no_improve, gate_checked)

        if epochs_no_improve >= patience:
            print(f'  Early stopping at epoch {epoch}'); break

    save_checkpoint(resume_ckpt, model, optimizer, scheduler,
                   epoch, best_val_acc, best_epoch, epochs_no_improve, gate_checked)

    elapsed = time.time() - start_time
    model.load_state_dict(torch.load(best_ckpt, weights_only=True))
    test_acc, test_f1 = evaluate_graph_level(model, test_loader, device)
    peak_mem = torch.cuda.max_memory_allocated()/1e9 if device.type == 'cuda' else 0.0

    print(f'\n  SEED {seed}: test_acc={test_acc:.4f} test_F1={test_f1:.4f} (val={best_val_acc:.4f}@ep{best_epoch})')
    print(f'  Time: {elapsed:.0f}s | Peak VRAM: {peak_mem:.2f} GB')

    result = {'seed': seed, 'test_acc': test_acc, 'test_f1': test_f1,
              'best_val_acc': best_val_acc, 'best_epoch': best_epoch,
              'time_s': elapsed, 'peak_gb': peak_mem}

    csv_path = 'results_hcp_task.csv'
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(result.keys()))
        if not file_exists: writer.writeheader()
        writer.writerow(result)
    print(f'  Result appended to {csv_path}')
    return result

## 11. Run Multi-Seed Training (Seed 0, 1, 2)

In [13]:
# Run Seed 0
res_0 = train_single_seed(
    seed=0,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    dropout=0.1, lr=1e-3, weight_decay=1e-4, max_epochs=200, patience=30, batch_size=32,
)

HCP-Task | Seed 0 | Hidden=128 | BS=32 | LR=0.001
Parameters: 361,992
No checkpoint — starting fresh.


  Ep   1 | loss=1.5894 | val_acc=0.5336 val_F1=0.4898 | best=0.5336@ep1 *BEST*


  Ep   2 | loss=0.7805 | val_acc=0.7097 val_F1=0.7020 | best=0.7097@ep2 *BEST*


  Ep   3 | loss=0.4699 | val_acc=0.8737 val_F1=0.8734 | best=0.8737@ep3 *BEST*


  Ep   4 | loss=0.3310 | val_acc=0.8844 val_F1=0.8850 | best=0.8844@ep4 *BEST*


...

  Ep  20 | loss=0.0356 | val_acc=0.9220 val_F1=0.9214 | best=0.9368@ep10


  Ep  25 | loss=0.0331 | val_acc=0.9341 val_F1=0.9347 | best=0.9368@ep10


  Ep  30 | loss=0.0377 | val_acc=0.9140 val_F1=0.9153 | best=0.9368@ep10


  Ep  35 | loss=0.0253 | val_acc=0.9247 val_F1=0.9253 | best=0.9368@ep10


  Ep  40 | loss=0.0330 | val_acc=0.9315 val_F1=0.9318 | best=0.9368@ep10
  Early stopping at epoch 40

  SEED 0: test_acc=0.9329 test_F1=0.9329 (val=0.9368@ep10)
  Time: 567s | Peak VRAM: 0.52 GB
  Result appended to results_hcp_task.csv


In [14]:
# Run Seed 1
res_1 = train_single_seed(
    seed=1,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    dropout=0.1, lr=1e-3, weight_decay=1e-4, max_epochs=200, patience=30, batch_size=32,
)

HCP-Task | Seed 1 | Hidden=128 | BS=32 | LR=0.001
Parameters: 361,992
No checkpoint — starting fresh.


  Ep   1 | loss=1.5975 | val_acc=0.5927 val_F1=0.5972 | best=0.5927@ep1 *BEST*


  Ep   2 | loss=0.7753 | val_acc=0.8105 val_F1=0.8120 | best=0.8105@ep2 *BEST*


  Ep   4 | loss=0.3735 | val_acc=0.8495 val_F1=0.8518 | best=0.8495@ep4 *BEST*


  Ep   5 | loss=0.3160 | val_acc=0.8884 val_F1=0.8885 | best=0.8884@ep5 *BEST*

--- Gate Health Check ---
  Layer 0: gate mean=0.5023, std=0.1834
  Layer 1: gate mean=0.5264, std=0.2610
  Layer 2: gate mean=0.5120, std=0.2671
  Layer 3: gate mean=0.4658, std=0.2461
-------------------------



...

  Ep  40 | loss=0.0290 | val_acc=0.9247 val_F1=0.9248 | best=0.9449@ep29


  Ep  45 | loss=0.0147 | val_acc=0.9382 val_F1=0.9383 | best=0.9449@ep29


  Ep  50 | loss=0.0267 | val_acc=0.9153 val_F1=0.9162 | best=0.9449@ep29


  Ep  55 | loss=0.0200 | val_acc=0.9315 val_F1=0.9314 | best=0.9449@ep29


  Early stopping at epoch 59

  SEED 1: test_acc=0.9342 test_F1=0.9342 (val=0.9449@ep29)
  Time: 848s | Peak VRAM: 0.52 GB
  Result appended to results_hcp_task.csv


In [15]:
# Run Seed 2
res_2 = train_single_seed(
    seed=2,
    in_dim=NODE_FEAT_DIM, hidden_dim=128, num_layers=4,
    num_classes=NUM_CLASSES, num_heads=4, lap_k=8, edge_dim=EDGE_DIM,
    dropout=0.1, lr=1e-3, weight_decay=1e-4, max_epochs=200, patience=30, batch_size=32,
)

HCP-Task | Seed 2 | Hidden=128 | BS=32 | LR=0.001
Parameters: 361,992
No checkpoint — starting fresh.


  Ep   1 | loss=1.5301 | val_acc=0.6546 val_F1=0.6449 | best=0.6546@ep1 *BEST*


  Ep   2 | loss=0.7149 | val_acc=0.7688 val_F1=0.7651 | best=0.7688@ep2 *BEST*


  Ep   3 | loss=0.4734 | val_acc=0.8145 val_F1=0.8144 | best=0.8145@ep3 *BEST*


  Ep   4 | loss=0.3571 | val_acc=0.8616 val_F1=0.8610 | best=0.8616@ep4 *BEST*


...

  Ep  95 | loss=0.0000 | val_acc=0.9476 val_F1=0.9477 | best=0.9503@ep81


  Ep 100 | loss=0.0000 | val_acc=0.9476 val_F1=0.9476 | best=0.9503@ep81


  Ep 105 | loss=0.0000 | val_acc=0.9449 val_F1=0.9451 | best=0.9503@ep81


  Ep 110 | loss=0.0000 | val_acc=0.9462 val_F1=0.9464 | best=0.9503@ep81


  Early stopping at epoch 111

  SEED 2: test_acc=0.9450 test_F1=0.9452 (val=0.9503@ep81)
  Time: 1585s | Peak VRAM: 0.52 GB
  Result appended to results_hcp_task.csv


## 12. Final Results Aggregation

In [16]:
csv_path = 'results_hcp_task.csv'
all_results = []
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        for row in csv.DictReader(f):
            all_results.append({k: float(v) if k != 'seed' else int(float(v)) for k, v in row.items()})

if all_results:
    accs = [r['test_acc'] for r in all_results]
    f1s  = [r['test_f1'] for r in all_results]
    n = len(all_results)
    print(f'HCP-Task | HybridGraphFNet (Graph-Level, GatedPooling)')
    for r in all_results:
        print(f'  Seed {r["seed"]}: Test Acc = {r["test_acc"]*100:.2f}% | Test Macro F1 = {r["test_f1"]:.4f} | Best Val = {r["best_val_acc"]*100:.2f}% (Ep {int(r["best_epoch"])})')
    print(f'\nSummary ({n} seeds):')
    print(f'  Test Accuracy: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%')
    print(f'  Test Macro F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
    print(f'\nLiterature Comparison (BrainMAP SOTA: 94.74% ± 0.07%):')
    print(f'  HybridGraphFNet vs BrainMAP: {np.mean(accs)*100 - 94.74:+.2f}%')
else:
    print('No results found yet in results_hcp_task.csv')

HCP-Task | HybridGraphFNet (Graph-Level, GatedPooling)
  Seed 0: Test Acc = 93.29% | Test Macro F1 = 0.9329 | Best Val = 93.68% (Ep 10)
  Seed 1: Test Acc = 93.42% | Test Macro F1 = 0.9342 | Best Val = 94.49% (Ep 29)
  Seed 2: Test Acc = 94.50% | Test Macro F1 = 0.9452 | Best Val = 95.03% (Ep 81)

Summary (3 seeds):
  Test Accuracy: 93.74% ± 0.54%
  Test Macro F1: 0.9374 ± 0.0055

Literature Comparison (BrainMAP SOTA: 94.74% ± 0.07%):
  HybridGraphFNet vs BrainMAP: -1.00%
